# Deploy Power BI Dashboard to Fabric

Reads the PBIP dashboard from the repo (uploaded to Lakehouse Files), patches
the Lakehouse SQL connection string, and deploys it as a Semantic Model + Report
in the Fabric workspace.

After deployment, saves `PBI_REPORT_ID` and `PBI_REPORT_LINK` to `.env` so that
`04_create_data_agent` can embed the dashboard link in the agent footer.

**Prerequisites:**
- Attach the Lakehouse
- Upload the `dashboard/` folder to Lakehouse Files (or run from a pipeline that has repo access)
- Run `01_download_cost_data` first so tables exist

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SEMANTIC_MODEL_NAME = "AzureBillingDashboard"
REPORT_NAME         = "AzureBillingDashboard"
REPORTS_FOLDER_NAME = "reports"

# Paths to PBIP files in Lakehouse Files
DASHBOARD_DIR  = "/lakehouse/default/Files/dashboard"
MODEL_BIM_PATH = f"{DASHBOARD_DIR}/AzureBillingDashboard.SemanticModel/model.bim"
PBISM_PATH     = f"{DASHBOARD_DIR}/AzureBillingDashboard.SemanticModel/definition.pbism"
REPORT_JSON    = f"{DASHBOARD_DIR}/AzureBillingDashboard.Report/report.json"
PBIR_PATH      = f"{DASHBOARD_DIR}/AzureBillingDashboard.Report/definition.pbir"
ENV_FILE       = "/lakehouse/default/Files/.env"

In [ ]:
# â”€â”€ Imports and Fabric context â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, requests, time, base64, os, re
from notebookutils import mssparkutils

fabric_token = mssparkutils.credentials.getToken("pbi")
headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json",
}
API = "https://api.fabric.microsoft.com/v1"

workspace_id = spark.conf.get("trident.workspace.id")
lakehouse_id = spark.conf.get("trident.lakehouse.id")
print(f"Workspace: {workspace_id}")
print(f"Lakehouse: {lakehouse_id}")

In [ ]:
# â”€â”€ Helper: poll long-running operation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def wait_for_operation(operation_id, label="Operation", max_wait=180):
    elapsed = 0
    while elapsed < max_wait:
        r = requests.get(f"{API}/operations/{operation_id}", headers=headers)
        r.raise_for_status()
        state = r.json()
        status = state.get("status", "Unknown")
        if status == "Succeeded":
            print(f"  {label}: Succeeded")
            return state
        if status == "Failed":
            raise RuntimeError(f"{label} failed: {json.dumps(state.get('error'), indent=2)}")
        time.sleep(5)
        elapsed += 5
    raise TimeoutError(f"{label} timed out after {max_wait}s")

def to_b64(text):
    """Base64-encode a string."""
    if isinstance(text, dict):
        text = json.dumps(text, ensure_ascii=False)
    return base64.b64encode(text.encode("utf-8")).decode("ascii")

In [ ]:
# â”€â”€ Resolve Lakehouse SQL endpoint and database â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Read from .env if available, otherwise derive from workspace items
lakehouse_endpoint = None
lakehouse_database = None

if os.path.isfile(ENV_FILE):
    with open(ENV_FILE, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("LAKEHOUSE_SQL_ENDPOINT="):
                lakehouse_endpoint = line.split("=", 1)[1].strip()
            elif line.startswith("LAKEHOUSE_DATABASE="):
                lakehouse_database = line.split("=", 1)[1].strip()

if not lakehouse_endpoint or not lakehouse_database:
    raise ValueError(
        "LAKEHOUSE_SQL_ENDPOINT and LAKEHOUSE_DATABASE must be set in .env. "
        "Run SetupDashboard.ps1 or set them manually."
    )

print(f"Lakehouse SQL Endpoint: {lakehouse_endpoint}")
print(f"Lakehouse Database: {lakehouse_database}")

In [ ]:
# â”€â”€ Read and patch model.bim â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open(MODEL_BIM_PATH, "r", encoding="utf-8") as f:
    model_bim = json.load(f)

# Patch the Lakehouse SQL Endpoint and Database parameter expressions
patched_params = 0
for expr in model_bim.get("model", {}).get("expressions", []):
    if expr.get("name") == "Lakehouse SQL Endpoint":
        new_expr = json.dumps(lakehouse_endpoint) + ' meta [IsParameterQuery=true, Type="Text", IsParameterQueryRequired=true]'
        if expr.get("expression") != new_expr:
            expr["expression"] = new_expr
            patched_params += 1
            print(f"  Patched 'Lakehouse SQL Endpoint' -> {lakehouse_endpoint}")
        else:
            print(f"  'Lakehouse SQL Endpoint' already correct")
    elif expr.get("name") == "Lakehouse Database":
        new_expr = json.dumps(lakehouse_database) + ' meta [IsParameterQuery=true, Type="Text", IsParameterQueryRequired=true]'
        if expr.get("expression") != new_expr:
            expr["expression"] = new_expr
            patched_params += 1
            print(f"  Patched 'Lakehouse Database' -> {lakehouse_database}")
        else:
            print(f"  'Lakehouse Database' already correct")

print(f"Patched {patched_params} parameter(s) in model.bim")
model_bim_json = json.dumps(model_bim, ensure_ascii=False, indent=2)

In [ ]:
# â”€â”€ Read definition.pbism â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open(PBISM_PATH, "r", encoding="utf-8") as f:
    pbism = f.read()
print(f"definition.pbism: {len(pbism)} chars")

In [ ]:
# â”€â”€ Resolve Reports folder â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
resp = requests.get(f"{API}/workspaces/{workspace_id}/folders", headers=headers)
resp.raise_for_status()
folders = resp.json().get("value", [])
reports_folder = next((f for f in folders if f["displayName"] == REPORTS_FOLDER_NAME), None)
reports_folder_id = reports_folder["id"] if reports_folder else None
print(f"Reports folder: {reports_folder_id or 'workspace root'}")

In [ ]:
# â”€â”€ Deploy Semantic Model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"Deploying Semantic Model: {SEMANTIC_MODEL_NAME}")

sm_parts = [
    {"path": "model.bim",        "payload": to_b64(model_bim_json), "payloadType": "InlineBase64"},
    {"path": "definition.pbism", "payload": to_b64(pbism),          "payloadType": "InlineBase64"},
]

# Check if SM already exists
resp = requests.get(f"{API}/workspaces/{workspace_id}/items?type=SemanticModel", headers=headers)
resp.raise_for_status()
existing_sm = next((i for i in resp.json()["value"] if i["displayName"] == SEMANTIC_MODEL_NAME), None)

if existing_sm:
    sm_id = existing_sm["id"]
    print(f"  Updating existing ({sm_id})")
    resp = requests.post(
        f"{API}/workspaces/{workspace_id}/items/{sm_id}/updateDefinition",
        headers=headers,
        json={"definition": {"format": "TMSL", "parts": sm_parts}},
    )
    if resp.status_code == 200:
        print("  Updated")
    elif resp.status_code == 202:
        wait_for_operation(resp.headers["x-ms-operation-id"], "Update SM")
    else:
        resp.raise_for_status()
else:
    print("  Creating new")
    create_body = {
        "displayName": SEMANTIC_MODEL_NAME,
        "type": "SemanticModel",
        "description": "Azure cost management semantic model",
        "definition": {"format": "TMSL", "parts": sm_parts},
    }
    if reports_folder_id:
        create_body["folderId"] = reports_folder_id
    resp = requests.post(
        f"{API}/workspaces/{workspace_id}/items",
        headers=headers,
        json=create_body,
    )
    if resp.status_code == 201:
        sm_id = resp.json()["id"]
        print(f"  Created ({sm_id})")
    elif resp.status_code == 202:
        wait_for_operation(resp.headers["x-ms-operation-id"], "Create SM")
        result = requests.get(
            f"{API}/operations/{resp.headers['x-ms-operation-id']}/result",
            headers=headers,
        )
        sm_id = result.json().get("id", "unknown")
        print(f"  Created ({sm_id})")
    else:
        resp.raise_for_status()

print(f"Semantic Model ID: {sm_id}")

In [ ]:
# â”€â”€ Read report.json and build definition.pbir pointing to the SM â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open(REPORT_JSON, "r", encoding="utf-8") as f:
    report_json_text = f.read()
print(f"report.json: {len(report_json_text):,} chars")

# Build definition.pbir that binds to the deployed semantic model by ID
pbir = {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definitionProperties/2.0.0/schema.json",
    "version": "4.0",
    "datasetReference": {
        "byConnection": {
            "connectionString": f"semanticmodelid={sm_id}",
        }
    },
}
print(f"definition.pbir -> semanticmodelid={sm_id}")

In [ ]:
# â”€â”€ Deploy Report â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"Deploying Report: {REPORT_NAME}")

report_parts = [
    {"path": "report.json",    "payload": to_b64(report_json_text), "payloadType": "InlineBase64"},
    {"path": "definition.pbir", "payload": to_b64(pbir),            "payloadType": "InlineBase64"},
]

# Check if report already exists
resp = requests.get(f"{API}/workspaces/{workspace_id}/items?type=Report", headers=headers)
resp.raise_for_status()
existing_rpt = next((i for i in resp.json()["value"] if i["displayName"] == REPORT_NAME), None)

if existing_rpt:
    rpt_id = existing_rpt["id"]
    print(f"  Updating existing ({rpt_id})")
    resp = requests.post(
        f"{API}/workspaces/{workspace_id}/items/{rpt_id}/updateDefinition",
        headers=headers,
        json={"definition": {"parts": report_parts}},
    )
    if resp.status_code == 200:
        print("  Updated")
    elif resp.status_code == 202:
        wait_for_operation(resp.headers["x-ms-operation-id"], "Update Report")
    else:
        resp.raise_for_status()
else:
    print("  Creating new")
    create_body = {
        "displayName": REPORT_NAME,
        "type": "Report",
        "description": "Azure cost management dashboard",
        "definition": {"parts": report_parts},
    }
    if reports_folder_id:
        create_body["folderId"] = reports_folder_id
    resp = requests.post(
        f"{API}/workspaces/{workspace_id}/items",
        headers=headers,
        json=create_body,
    )
    if resp.status_code == 201:
        rpt_id = resp.json()["id"]
        print(f"  Created ({rpt_id})")
    elif resp.status_code == 202:
        wait_for_operation(resp.headers["x-ms-operation-id"], "Create Report")
        result = requests.get(
            f"{API}/operations/{resp.headers['x-ms-operation-id']}/result",
            headers=headers,
        )
        rpt_id = result.json().get("id", "unknown")
        print(f"  Created ({rpt_id})")
    else:
        resp.raise_for_status()

report_link = f"https://app.powerbi.com/groups/{workspace_id}/reports/{rpt_id}?experience=power-bi"
print(f"\nReport ID: {rpt_id}")
print(f"Report Link: {report_link}")

In [ ]:
# â”€â”€ Save report details to .env â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Updating .env with report details...")

env_lines = []
if os.path.isfile(ENV_FILE):
    with open(ENV_FILE, "r", encoding="utf-8") as f:
        env_lines = f.readlines()

# Remove existing PBI_ entries
env_lines = [l for l in env_lines if not l.strip().startswith("PBI_REPORT_ID=")
             and not l.strip().startswith("PBI_REPORT_LINK=")
             and not l.strip().startswith("PBI_SEMANTIC_MODEL_ID=")]

# Ensure trailing newline before appending
if env_lines and not env_lines[-1].endswith("\n"):
    env_lines[-1] += "\n"

# Add new entries
env_lines.append("\n")
env_lines.append("# Power BI Dashboard (set by 02_deploy_dashboard)\n")
env_lines.append(f"PBI_SEMANTIC_MODEL_ID={sm_id}\n")
env_lines.append(f"PBI_REPORT_ID={rpt_id}\n")
env_lines.append(f"PBI_REPORT_LINK={report_link}\n")

with open(ENV_FILE, "w", encoding="utf-8") as f:
    f.writelines(env_lines)

print(f"  PBI_SEMANTIC_MODEL_ID={sm_id}")
print(f"  PBI_REPORT_ID={rpt_id}")
print(f"  PBI_REPORT_LINK={report_link}")
print("\n.env updated. The 04_create_data_agent notebook will use PBI_REPORT_LINK for the dashboard footer.")

In [ ]:
# â”€â”€ Summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from datetime import datetime

print(f"\n{'='*60}")
print(f"Dashboard deployment completed: {datetime.utcnow().isoformat()}Z")
print(f"{'='*60}")
print(f"  Semantic Model : {SEMANTIC_MODEL_NAME} ({sm_id})")
print(f"  Report         : {REPORT_NAME} ({rpt_id})")
print(f"  Folder         : {REPORTS_FOLDER_NAME}")
print(f"  Lakehouse      : {lakehouse_database} @ {lakehouse_endpoint}")
print(f"  Report URL     : {report_link}")
print(f"{'='*60}")